# MoE synthetic augmentation experiment

Continue the published ModernBERT MoE checkpoint with 976 synthetic training responses
and 1,952 fixed original-training replay examples. The original labels are from Writing9;
the synthetic labels are LLM judgments. Neither is examiner ground truth.

## Method

Train all encoder and head parameters for at most five epochs, using separate learning
rates of 5e-6 and 5e-5. Select using original and synthetic validation only, with an
original-data regression guard. Freeze selection before the final comparison.
Both model and dataset downloads use immutable Hugging Face revisions. The attached
private original dataset must match its pinned checksums. No public upload occurs here.


## Restore the reviewed code and pinned dependencies

In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
project = pathlib.Path("/kaggle/temp/synthetic-moe-code")
project.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '46fab05e09f4d4e46537f39c99c30c2ba8144d0f8ef11270c0ee537df76d21d9'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(project)
os.chdir(project)
dependencies = "/kaggle/temp/synthetic-moe-dependencies"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", dependencies,
                "transformers==4.57.6", "huggingface-hub==0.36.0", "tokenizers==0.22.2", "safetensors==0.7.0"], check=True)
os.environ.update(PYTHONPATH=dependencies + os.pathsep + str(project / "src"),
                  HF_HOME="/kaggle/temp/huggingface", HF_HUB_DISABLE_TELEMETRY="1",
                  TOKENIZERS_PARALLELISM="false", WANDB_DISABLED="true", CUDA_VISIBLE_DEVICES="0")
pathlib.Path("/kaggle/working/source_bundle_sha256.txt").write_text('46fab05e09f4d4e46537f39c99c30c2ba8144d0f8ef11270c0ee537df76d21d9')
print("Code SHA256:", '46fab05e09f4d4e46537f39c99c30c2ba8144d0f8ef11270c0ee537df76d21d9')
print(json.dumps(json.loads((project / "configs/synthetic_moe.json").read_text()), indent=2))


## Train and evaluate

Detailed per-epoch logs and prediction tables are saved as private outputs.

In [ ]:
command = [sys.executable, "-m", "ielts_scoring.synthetic_finetune",
           "--config", "configs/synthetic_moe.json", "--data-root", "/kaggle/input",
           "--output", "/kaggle/working"]
with open("/kaggle/working/training.log", "w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line)
        log.flush()
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Training failed. See training.log.")


## Results

Lower macro MAE is better. These are label-source-specific comparisons, not examiner accuracy.

In [ ]:
import pandas as pd
result = json.loads(pathlib.Path("/kaggle/working/result.json").read_text())
if result["status"] == "evaluated":
    display(pd.DataFrame([
        {"Evaluation": name, "Parent macro MAE": result["baseline"][name]["macro_mae"],
         "Candidate macro MAE": values["macro_mae"],
         "Change": values["macro_mae"] - result["baseline"][name]["macro_mae"]}
        for name, values in result["candidate"].items()
    ]).round(4))
print(json.dumps(result["promotion"], indent=2))
